# Activity Prediction (Phase 3) — Docking & Molecular Dynamics

Self-contained pipeline for docking and molecular dynamics of COX-2 inhibitor
cluster representatives. Runs docking locally via subprocess (canonical reproducible
path) with optional SLURM submission as an execution backend.

**Execution contract:**
- Run cells top-to-bottom. Each stage explicitly defines its output variables.
- The state-init cell guarantees all intermediate variables exist,
  making the notebook restart-safe from any cell.
- Skipping stages is safe: downstream cells guard on variable state.

**Inputs:**
- `mol_files/8. Clustering/Thiazolones_10_samples.csv` (ID, SMILES)
- `protein_files/pdb/6COX.pdb` (COX-2)
- `protein_files/pdb/3KK6.pdb` (COX-1)

**Workflow:**
1. **PREPARE** — Ligand + receptor preparation, binding site detection
2. **RUN** — Docking execution (local or SLURM backend)
3. **PARSE** — Validation, score extraction, composite ranking, MD candidate selection

**Note:** Docking scores are a ranking proxy, not an energy model.
COX-2 and COX-1 scores are normalized within their own distributions before comparison.
MM-GBSA is available as a standalone optional tool but is NOT used in ranking.

## 0. Setup

In [1]:
from pathlib import Path

import pandas as pd

import py_utils as pu
import py_utils.docking as dock

print(f"py_utils {pu.__version__}")

RUN_MODE = "local"       # "local" or "slurm"
FORCE_RUN_DOCKING = False  # True = delete old results for current mapping and re-dock
SUBMIT_SLURM = False       # True = submit SLURM job via sbatch subprocess

# Pipeline constants
SEED = 42
BOX_SIZE = 24.0
RESCORE_FRACTION = 0.25
N_SNAPSHOTS = 100
PRODUCTION_NS = 2

# SLURM configuration (adjust to your cluster)
SLURM_PARTITION = "normal"
SLURM_ACCOUNT = None

# Paths
ROOT = Path.cwd()
INPUT_CSV = ROOT / "mol_files/8. Clustering/Thiazolones_10_samples.csv"
COX2_PDB = ROOT / "protein_files/pdb/6COX.pdb"
COX1_PDB = ROOT / "protein_files/pdb/3KK6.pdb"
COX2_LIGAND_RESNAME = "S58"
COX1_LIGAND_RESNAME = "FLC"

py_utils 5.5


## 0. Create directories

In [2]:
dirs = dock.init_hpc_dirs(ROOT / "hpc_jobs")

[init_hpc_dirs] Created 13 directories under /Users/dariomlorente/code/coxib-drug-design/hpc_jobs


## 0b. Initialize pipeline state (restart-safe)

In [3]:
# All intermediate variables initialized to safe defaults.
# Each stage below overwrites these deterministically.
# This guarantees no NameError regardless of execution order.
df_ligands_raw = pd.DataFrame()
df_ligands = pd.DataFrame()
cox2_info = {}
cox1_info = {}
mapping_path = None
mapping_df = pd.DataFrame()
docking_script = None
df_scores = pd.DataFrame()
df_instability = pd.DataFrame()
df_analysis = pd.DataFrame()
df_ranked = pd.DataFrame()
df_rescore = pd.DataFrame()
rescore_ids = []
rescore_script = None
df_mmgbsa = pd.DataFrame()
md_candidate_ids = []
md_script = None
docking_validation = {}


## 1. Prepare ligands and receptors

In [4]:
df_ligands_raw = pd.read_csv(INPUT_CSV)
print(f"[Load ligands] {len(df_ligands_raw)} compounds from {INPUT_CSV.name}")
display(df_ligands_raw[["ID", "SMILES"]].head())

df_ligands = dock.prepare_ligands(
    df_ligands_raw,
    ligands_dir=dirs["ligands"],
    seed=SEED,
)

[Load ligands] 10 compounds from Thiazolones_10_samples.csv


,ID,SMILES
0,A40C14S,COc1ccc(/C=C2\N=C(c3ccnc4ccccc34)SC2=O)c(OC)c1OC
1,A60C24S,COc1ccccc1/C=C1\N=C(c2c(F)cccc2Br)SC1=O
2,A89C27S,C=C(C)/C=C1\N=C(c2ccccn2)SC1=O
3,A47C32S,COc1ccc(C2=N/C(=C\c3cccnc3)C(=O)S2)c(Br)c1
4,A34C7S,COc1ccc(C2=N/C(=C\c3ccc[nH]3)C(=O)S2)cc1OC


[prepare_ligands] 10/10 ligands prepared successfully


In [5]:
# 6COX (COX-2): binding site from co-crystallized SC-558 (S58)
cox2_info = dock.prepare_receptor(
    pdb_path=COX2_PDB,
    receptor_dir=dirs["receptors"],
    receptor_id="6COX",
    ligand_resname=COX2_LIGAND_RESNAME,
    box_size=BOX_SIZE,
)

print(f"\nCOX-2 box center: {cox2_info['box_center']}")

if cox2_info:
    # 3KK6 (COX-1): binding site from co-crystallized celecoxib (FLC)
    cox1_info = dock.prepare_receptor(
        pdb_path=COX1_PDB,
        receptor_dir=dirs["receptors"],
        receptor_id="3KK6",
        ligand_resname=COX1_LIGAND_RESNAME,
        box_size=BOX_SIZE,
    )

    print(f"\nCOX-1 box center (from {COX1_LIGAND_RESNAME} ligand): {cox1_info['box_center']}")
else:
    print("[COX-1 preparation] COX-2 receptor not prepared first")

[prepare_receptor] Generated /Users/dariomlorente/code/coxib-drug-design/hpc_jobs/receptors/6COX.pdbqt
[get_binding_site_center] Using S58 centroid from 6COX.pdb
[prepare_receptor] Saved /Users/dariomlorente/code/coxib-drug-design/hpc_jobs/receptors/6COX_box.json

COX-2 box center: [46.87599999999999, 18.92428846153846, 44.33801923076922]
[prepare_receptor] Generated /Users/dariomlorente/code/coxib-drug-design/hpc_jobs/receptors/3KK6.pdbqt
[get_binding_site_center] Using FLC centroid from 3KK6.pdb
[prepare_receptor] Saved /Users/dariomlorente/code/coxib-drug-design/hpc_jobs/receptors/3KK6_box.json

COX-1 box center (from FLC ligand): [-41.060722222222225, 66.59022222222224, -1.4707777777777777]


In [6]:
import json

for rec_id, info in [("6COX", cox2_info), ("3KK6", cox1_info)]:
    if not info:
        print(f"\n{rec_id}: not prepared — run receptor preparation first")
        continue
    box_path = dirs["receptors"] / f"{rec_id}_box.json"
    with open(box_path) as f:
        box = json.load(f)
    print(f"\n{rec_id}:")
    print(f"  Center: ({box['center_x']}, {box['center_y']}, {box['center_z']})")
    print(f"  Size: {box['size_x']} x {box['size_y']} x {box['size_z']} Å")


6COX:
  Center: (46.876, 18.924, 44.338)
  Size: 24.0 x 24.0 x 24.0 Å

3KK6:
  Center: (-41.061, 66.59, -1.471)
  Size: 24.0 x 24.0 x 24.0 Å


In [7]:
if len(df_ligands) > 0:
    receptor_ids = ["6COX", "3KK6"]

    mapping_path = dock.write_mapping_csv(
        df_ligands=df_ligands,
        receptor_ids=receptor_ids,
        mapping_csv=dirs["docking_inputs"] / "mapping.csv",
    )

    mapping_df = pd.read_csv(mapping_path)
    display(mapping_df.head(10))
    print(f"\nTotal docking tasks: {len(mapping_df)}")
else:
    print("[Docking mapping] No ligands available — run prepare_ligands first")

[write_mapping_csv] 20 docking tasks written to /Users/dariomlorente/code/coxib-drug-design/hpc_jobs/docking/inputs/mapping.csv


,task_id,ligand_id,receptor_id
0,1,A40C14S,6COX
1,2,A40C14S,3KK6
2,3,A60C24S,6COX
3,4,A60C24S,3KK6
4,5,A89C27S,6COX
5,6,A89C27S,3KK6
6,7,A47C32S,6COX
7,8,A47C32S,3KK6
8,9,A34C7S,6COX
9,10,A34C7S,3KK6



Total docking tasks: 20


## 2. Run docking

In [8]:
if RUN_MODE != "local":
    print("[Docking local] Skipped — RUN_MODE is not 'local'")
else:
    import shutil
    import time
    import subprocess
    import json

    # Check if Vina is available
    vina_exe = shutil.which("vina")
    if vina_exe is None:
        raise RuntimeError(
            "AutoDock Vina not found. Install with: "
            "conda install -n docking -c conda-forge vina=1.2.5"
        )
    print(f"[Docking local] Using Vina at: {vina_exe}")

    if mapping_path is None or not mapping_path.exists():
        print("[Docking local] No mapping file — run docking mapping first")
    else:
        mapping_df = pd.read_csv(mapping_path)
        logs_dir = dirs["docking_logs"]
        results_dir = dirs["docking_results"]
        ligands_dir = dirs["ligands"]
        receptors_dir = dirs["receptors"]

        # Force re-docking: delete only files associated with current mapping
        if FORCE_RUN_DOCKING:
            n_deleted = 0
            for _, task in mapping_df.iterrows():
                lid, rid = task["ligand_id"], task["receptor_id"]
                for f in [logs_dir / f"{lid}_{rid}.log", results_dir / f"{lid}_{rid}_out.pdbqt"]:
                    if f.exists():
                        f.unlink()
                        n_deleted += 1
            print(f"[Docking local] Deleted {n_deleted} files (FORCE_RUN_DOCKING)")
        elif not any(d.exists() for d in [logs_dir, results_dir]):
            print("[Docking local] No previous results — running fresh docking")

        n_total = len(mapping_df)
        n_done = 0
        n_skipped = 0
        n_failed = 0
        start = time.time()

        for _, task in mapping_df.iterrows():
            ligand_id = task["ligand_id"]
            receptor_id = task["receptor_id"]

            log_path = logs_dir / f"{ligand_id}_{receptor_id}.log"
            pose_path = results_dir / f"{ligand_id}_{receptor_id}_out.pdbqt"

            if log_path.exists() and pose_path.exists():
                n_skipped += 1
                continue

            ligand_pdbqt = ligands_dir / f"{ligand_id}.pdbqt"
            receptor_pdbqt = receptors_dir / f"{receptor_id}.pdbqt"
            box_json = receptors_dir / f"{receptor_id}_box.json"

            if not ligand_pdbqt.exists() or not receptor_pdbqt.exists():
                print(f"[Docking local] Missing input for {ligand_id} x {receptor_id}")
                n_failed += 1
                continue

            with open(box_json) as f:
                box = json.load(f)

            cmd = [
                vina_exe,
                "--receptor", str(receptor_pdbqt),
                "--ligand", str(ligand_pdbqt),
                "--center_x", str(box["center_x"]),
                "--center_y", str(box["center_y"]),
                "--center_z", str(box["center_z"]),
                "--size_x", str(box["size_x"]),
                "--size_y", str(box["size_y"]),
                "--size_z", str(box["size_z"]),
                "--cpu", "4",
                "--exhaustiveness", "16",
                "--num_modes", "20",
                "--energy_range", "5",
                "--out", str(pose_path),
            ]

            result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
            with open(log_path, "w") as f:
                f.write(result.stdout)
                if result.stderr:
                    f.write(result.stderr)

            if result.returncode == 0:
                n_done += 1
                print(f"  [{n_done + n_skipped}/{n_total}] {ligand_id} x {receptor_id} done")
            else:
                n_failed += 1
                print(f"  FAILED: {ligand_id} x {receptor_id}: {result.stderr[:200]}")

        elapsed = time.time() - start
        print(f"\n[Docking local] {n_done} completed, {n_skipped} skipped, {n_failed} failed")
        print(f"[Docking local] Total time: {elapsed/60:.1f} min")


[Docking local] Using Vina at: /Users/dariomlorente/miniconda3/envs/docking/bin/vina
  [1/20] A40C14S x 6COX done
  [2/20] A40C14S x 3KK6 done
  [3/20] A60C24S x 6COX done
  [4/20] A60C24S x 3KK6 done
  [5/20] A89C27S x 6COX done
  [6/20] A89C27S x 3KK6 done
  [7/20] A47C32S x 6COX done
  [8/20] A47C32S x 3KK6 done
  [9/20] A34C7S x 6COX done
  [10/20] A34C7S x 3KK6 done
  [11/20] A42C30S x 6COX done
  [12/20] A42C30S x 3KK6 done
  [13/20] A70C29S x 6COX done
  [14/20] A70C29S x 3KK6 done
  [15/20] A78C7S x 6COX done
  [16/20] A78C7S x 3KK6 done
  [17/20] A70C39S x 6COX done
  [18/20] A70C39S x 3KK6 done
  [19/20] A13C37S x 6COX done
  [20/20] A13C37S x 3KK6 done

[Docking local] 20 completed, 0 skipped, 0 failed
[Docking local] Total time: 1.8 min


In [9]:
if RUN_MODE != "slurm":
    print("[Docking SLURM] Skipped — RUN_MODE is not 'slurm'")
else:
    if mapping_path is not None and mapping_path.exists():
        mapping_df_slurm = pd.read_csv(mapping_path)
        n_tasks = len(mapping_df_slurm)
        docking_script = dock.generate_docking_slurm(
            mapping_csv=mapping_path,
            slurm_dir=dirs["docking_slurm"],
            n_tasks=n_tasks,
            base_dir=ROOT / "hpc_jobs",
            partition=SLURM_PARTITION,
            account=SLURM_ACCOUNT,
        )
        print(f"\nSubmit with: sbatch {docking_script}")

        if SUBMIT_SLURM:
            try:
                result = subprocess.run(
                    ["sbatch", str(docking_script)],
                    capture_output=True, text=True, timeout=30,
                )
                if result.returncode == 0:
                    job_id = result.stdout.strip().split()[-1]
                    print(f"[Docking SLURM] Submitted job {job_id}")
                else:
                    print(f"[Docking SLURM] sbatch failed: {result.stderr.strip()}")
            except FileNotFoundError:
                print("[Docking SLURM] sbatch not found — not on an HPC cluster")
        else:
            print("[Docking SLURM] Set SUBMIT_SLURM = True to submit automatically")
    else:
        print("[Docking SLURM] No mapping file — run docking mapping first")


[Docking SLURM] Skipped — RUN_MODE is not 'slurm'


## 3. Validate and parse results

In [10]:
if mapping_path is None or not mapping_path.exists():
    print("[Validate docking] No mapping file — run docking mapping first")
    docking_validation = {"status": "FAIL", "summary": "No mapping file"}
else:
    docking_validation = dock.validate_docking_results(
        mapping_csv=mapping_path,
        logs_dir=dirs["docking_logs"],
        results_dir=dirs["docking_results"],
    )
    if docking_validation["status"] == "FAIL":
        print(f"⚠️  No valid docking results. Run docking (local or SLURM) first.")
    elif docking_validation["status"] == "PARTIAL":
        print(f"⚠️  Partial results: {docking_validation['complete']}/{docking_validation['total']} tasks validated.")
        print("   Downstream analysis will use only validated tasks.")


[validate_docking] PASS: 20/20 docking tasks validated. Missing: 0, Failed: 0, Orphaned: 0


In [11]:
if docking_validation.get("status") in ("PASS", "PARTIAL"):
    df_scores, df_instability = dock.parse_docking_logs(
        results_dir=dirs["docking_results"],
        logs_dir=dirs["docking_logs"],
        receptor_map={"6COX": "COX2", "3KK6": "COX1"},
    )

    if not df_scores.empty:
        display(df_scores.head())
        print(f"\n[Docking scores] {len(df_scores)} results parsed")

    if not df_instability.empty:
        display(df_instability.head())
        print(f"\n[Instability] {len(df_instability)} entries parsed")
else:
    df_scores = pd.DataFrame()
    df_instability = pd.DataFrame()
    print(f"[Parse docking] Skipped — validation status: {docking_validation.get('status')}")


[parse_docking_logs] Parsed 20 docking results


,ligand_id,receptor_id,cox_label,docking_score
0,A42C30S,3KK6,COX1,-8.48
1,A40C14S,3KK6,COX1,-8.30
2,A89C27S,3KK6,COX1,-6.42
3,A78C7S,3KK6,COX1,-8.69
4,A60C24S,3KK6,COX1,-7.49



[Docking scores] 20 results parsed


,ligand_id,receptor_id,cox_label,pose_spread
0,A42C30S,3KK6,COX1,10.400
1,A40C14S,3KK6,COX1,8.919
2,A89C27S,3KK6,COX1,5.581
3,A78C7S,3KK6,COX1,5.189
4,A60C24S,3KK6,COX1,7.141



[Instability] 20 entries parsed


In [12]:
if not df_scores.empty and not df_instability.empty:
    df_analysis = dock.compute_docking_analysis(df_scores, df_instability)
    if not df_analysis.empty:
        display(df_analysis)
else:
    df_analysis = pd.DataFrame()
    print("[Docking analysis] No docking results available — run docking on HPC first")

cox_label,ligand_id,score_cox1,score_cox2,instability_cox2,instability_cox1,instability
0,A13C37S,-7.70,-8.74,19.090,7.061,19.090
1,A34C7S,-7.27,-7.70,8.467,8.148,8.467
2,A40C14S,-8.30,-8.11,22.180,8.919,22.180
3,A42C30S,-8.48,-9.50,8.717,10.400,10.400
4,A47C32S,-7.69,-7.95,16.480,6.674,16.480
5,A60C24S,-7.49,-7.94,17.130,7.141,17.130
6,A70C29S,-8.09,-7.99,18.630,4.800,18.630
7,A70C39S,-8.40,-8.13,18.090,4.688,18.090
8,A78C7S,-8.69,-7.69,23.300,5.189,23.300
9,A89C27S,-6.42,-6.62,16.430,5.581,16.430


In [13]:
if not df_analysis.empty:
    dock.save_docking_scores(
        df_analysis,
        output_path=ROOT / "protein_files/docking/docking_scores.csv",
    )
else:
    print("[Save docking] No analysis data to save")

[save_docking_scores] Saved 10 entries to /Users/dariomlorente/code/coxib-drug-design/protein_files/docking/docking_scores.csv


In [14]:
if not df_analysis.empty and "score_cox2" in df_analysis.columns:
    n_rescore = max(1, int(len(df_analysis) * RESCORE_FRACTION))
    df_rescore = df_analysis.nsmallest(n_rescore, "score_cox2")
    rescore_ids = df_rescore["ligand_id"].tolist()
    print(f"[MM-GBSA selection] Top {n_rescore} compounds by COX-2 docking score:")
    print(f"  IDs: {', '.join(rescore_ids)}")
else:
    df_rescore = pd.DataFrame()
    rescore_ids = []
    print("[MM-GBSA selection] No docking results available for selection")

[MM-GBSA selection] Top 2 compounds by COX-2 docking score:
  IDs: A42C30S, A13C37S


In [15]:
if rescore_ids:
    n_tasks = len(rescore_ids)
    rescore_script = dock.generate_rescore_slurm(
        slurm_dir=dirs["rescoring_slurm"],
        n_tasks=n_tasks,
        base_dir=ROOT / "hpc_jobs",
        partition=SLURM_PARTITION,
        account=SLURM_ACCOUNT,
        n_snapshots=N_SNAPSHOTS,
    )
    print(f"\nSubmit with: sbatch {rescore_script}")
else:
    rescore_script = None
    print("[MM-GBSA SLURM] No compounds selected for rescoring")


[generate_rescore_slurm] Written /Users/dariomlorente/code/coxib-drug-design/hpc_jobs/rescoring/slurm/mmgbsa_array.sh

Submit with: sbatch /Users/dariomlorente/code/coxib-drug-design/hpc_jobs/rescoring/slurm/mmgbsa_array.sh


In [16]:
df_mmgbsa = dock.parse_mmgbsa_results(dirs["rescoring_results"])

if not df_mmgbsa.empty:
    display(df_mmgbsa)
    dock.save_mmgbsa_scores(df_mmgbsa)
else:
    df_mmgbsa = pd.DataFrame()
    print("[MM-GBSA results] No results available — run rescoring on HPC first")

[parse_mmgbsa_results] No MM-GBSA results found
[MM-GBSA results] No results available — run rescoring on HPC first


In [17]:
if not df_analysis.empty:
    df_ranked = df_analysis.copy()
    print(f"[Ranking] {len(df_ranked)} compounds ready for scoring")
else:
    df_ranked = pd.DataFrame()
    print("[Ranking] No docking analysis data available")

if len(df_ranked) > 0:
    # MD_score = 2 * norm_rank(COX2) - 1 * norm_rank(COX1) - λ * instability
    # COX-2 and COX-1 scores are normalized within their own distributions.
    # Docking scores are a ranking proxy, not an energy model.
    df_ranked = dock.compute_composite_score(
        df_ranked,
        instability_lambda=0.2,
    )
else:
    print("[Composite scoring] No data to score")

[Ranking] 10 compounds ready for scoring
[compute_composite_score] MD scores computed for 10 compounds
[compute_composite_score] Formula: MD_score = 2*norm_rank(COX2) - 1*norm_rank(COX1) - λ*instability
[compute_composite_score] Instability penalty λ = 0.2


In [18]:
if len(df_ranked) > 0:
    display_cols = [
        "ligand_id", "score_cox2", "score_cox1",
        "instability_cox2", "instability_cox1", "instability",
        "norm_rank_cox2", "norm_rank_cox1", "instability_penalty",
        "md_score",
    ]
    available_cols = [c for c in display_cols if c in df_ranked.columns]
    display(df_ranked[available_cols].round(3))
else:
    print("[Ranking table] No ranked data available")

cox_label,ligand_id,score_cox2,score_cox1,instability_cox2,instability_cox1,instability,norm_rank_cox2,norm_rank_cox1,instability_penalty,md_score
0,A42C30S,-9.50,-8.48,8.717,10.400,10.400,1.000,0.889,2.080,-0.969
1,A34C7S,-7.70,-7.27,8.467,8.148,8.467,0.222,0.111,1.693,-1.360
2,A13C37S,-8.74,-7.70,19.090,7.061,19.090,0.889,0.444,3.818,-2.485
3,A47C32S,-7.95,-7.69,16.480,6.674,16.480,0.444,0.333,3.296,-2.740
4,A70C39S,-8.13,-8.40,18.090,4.688,18.090,0.778,0.778,3.618,-2.840
5,A60C24S,-7.94,-7.49,17.130,7.141,17.130,0.333,0.222,3.426,-2.982
6,A70C29S,-7.99,-8.09,18.630,4.800,18.630,0.556,0.556,3.726,-3.170
7,A89C27S,-6.62,-6.42,16.430,5.581,16.430,0.000,0.000,3.286,-3.286
8,A40C14S,-8.11,-8.30,22.180,8.919,22.180,0.667,0.667,4.436,-3.769
9,A78C7S,-7.69,-8.69,23.300,5.189,23.300,0.111,1.000,4.660,-5.438


In [19]:
if len(df_ranked) > 0:
    md_candidate_ids = dock.select_md_candidates(
        df_ranked,
        n=5,
        max_instability=30.0,
    )
    print(f"[MD candidates] {len(md_candidate_ids)} selected:")
    for i, cid in enumerate(md_candidate_ids, 1):
        row = df_ranked[df_ranked["ligand_id"] == cid].iloc[0]
        score = row.get("md_score", "N/A")
        inst = row.get("instability", "N/A")
        print(f"  {i}. {cid} (md_score={score}, instability={inst})")
else:
    md_candidate_ids = []
    print("[MD candidates] No ranked data available")

[MD candidates] 5 selected:
  1. A42C30S (md_score=-0.9689, instability=10.4)
  2. A34C7S (md_score=-1.3601, instability=8.467)
  3. A13C37S (md_score=-2.4847, instability=19.09)
  4. A47C32S (md_score=-2.7404, instability=16.48)
  5. A70C39S (md_score=-2.8402, instability=18.09)


In [20]:
if md_candidate_ids:
    dock.save_md_candidates(
        md_candidate_ids,
        output_path=dirs["md_candidates"] / "md_candidates.txt",
    )
else:
    print("[Save candidates] No candidates to save")

[save_md_candidates] Saved 5 candidates to /Users/dariomlorente/code/coxib-drug-design/hpc_jobs/md/candidates/md_candidates.txt


In [21]:
candidates_file = dirs["md_candidates"] / "md_candidates.txt"
if candidates_file.exists() and md_candidate_ids:
    md_script = dock.generate_md_slurm(
        candidates_txt=candidates_file,
        slurm_dir=dirs["md_slurm"],
        base_dir=ROOT / "hpc_jobs",
        partition=SLURM_PARTITION,
        account=SLURM_ACCOUNT,
        production_ns=PRODUCTION_NS,
    )
    print(f"\nSubmit with: sbatch {md_script}")
else:
    md_script = None
    print("[MD SLURM] No candidates available — run MD candidate selection first")

[generate_md_slurm] Written /Users/dariomlorente/code/coxib-drug-design/hpc_jobs/md/slurm/md_array.sh for 5 candidates

Submit with: sbatch /Users/dariomlorente/code/coxib-drug-design/hpc_jobs/md/slurm/md_array.sh


In [22]:
md_records = []
md_systems_dir = dirs["md_systems"]

for candidate_dir in sorted(md_systems_dir.iterdir()):
    if not candidate_dir.is_dir():
        continue
    results_dir = candidate_dir / "results"
    rmsd_file = results_dir / "rmsd.xvg"
    if not rmsd_file.exists():
        continue

    try:
        times = []
        rmsds = []
        with open(rmsd_file) as f:
            for line in f:
                line = line.strip()
                if line.startswith(("#", "@")):
                    continue
                parts = line.split()
                if len(parts) >= 2:
                    times.append(float(parts[0]))
                    rmsds.append(float(parts[1]))

        if rmsds:
            import numpy as np
            md_records.append({
                "ligand_id": candidate_dir.name,
                "rmsd_mean_nm": round(float(np.mean(rmsds)), 3),
                "rmsd_std_nm": round(float(np.std(rmsds)), 3),
                "rmsd_max_nm": round(float(np.max(rmsds)), 3),
                "rmsd_min_nm": round(float(np.min(rmsds)), 3),
                "frame_count": len(rmsds),
            })
    except Exception:
        continue

if md_records:
    df_md = pd.DataFrame(md_records)
    md_out = ROOT / "protein_files/md/md_stability.csv"
    md_out.parent.mkdir(parents=True, exist_ok=True)
    df_md.to_csv(md_out, index=False)
    print(f"[MD analysis] Saved {len(df_md)} entries to {md_out}")
    display(df_md)
else:
    print("[MD analysis] No RMSD files found")


[MD analysis] No RMSD files found
